# Text as Data: dos hipótesis, un corpus
Doctorado en Ciencia Política — UNMSM · Métodos Cuantitativos (2026)

**Puzzle:** ¿Hasta qué punto los partidos convergen en sus propuestas pese a diferenciarse ideológicamente?

**Unidad de análisis:** un plan completo por partido. En este notebook prepararemos un corpus con una fila por plan.

Pregunta → hipótesis → evidencia → representación → técnica. La preparación organiza la evidencia textual; todavía no evalúa las hipótesis. El análisis posterior buscará describir e interpretar diferencias discursivas, no predecir elecciones ni establecer causas por sí solo.

El recorrido de este notebook es:

**PDFs en memoria → una fila por plan → limpieza mínima → comparación → un CSV.**

Ejecute los bloques en orden. Los ocho planes ficticios se leen directamente desde GitHub; no necesita cargar los PDFs ni guardarlos en una carpeta local. Solo al final se guarda el corpus preparado para utilizarlo en `TextAnalysis.ipynb`.

## preTEXT-01: definir los documentos del corpus

Instalamos **PyMuPDF**, la biblioteca que utilizaremos para extraer texto de los PDFs. Luego definimos la dirección del repositorio y los datos de los ocho partidos.

`MANIFIESTO` reúne los identificadores, nombres, candidaturas, nombres de archivo y enlaces directos a los PDFs. Es una lista de documentos y sus datos de identificación; todavía no contiene el texto de los planes. En este bloque no se leen ni se guardan PDFs.

**Clave:** los identificadores permiten mantener la correspondencia entre cada partido y su plan durante todo el análisis.

In [6]:
!pip -q install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 37.7 MB/s eta 0:00:00


In [7]:
from pathlib import Path
import csv, json, re, hashlib, unicodedata
from urllib.request import urlopen



BASE_URL = 'https://raw.githubusercontent.com/doctorado-cuanticp/text/main/planes_gobierno'

# Cambie solo estos datos si los partidos reales tienen otro nombre o candidatura.
# No cambie el orden ni las identidades para obtener un resultado esperado.
PARTIDOS = [
    ('A', 'Renovación Popular Social', 'Candidata A'),
    ('B', 'Frente Progresista',        'Candidato B'),
    ('C', 'Movimiento Democrático',    'Candidata C'),
    ('D', 'Centro Cívico',             'Candidato D'),
    ('E', 'Alianza Nacional',          'Candidata E'),
    ('F', 'Libertad Republicana',      'Candidato F'),
    ('G', 'Futuro Liberal',            'Candidata G'),
    ('H', 'Mercado y Libertad',        'Candidato H'),
]

MANIFIESTO = [
    {
        'id_partido': id_,
        'partido': partido,
        'candidato': candidato,
        'archivo': f'Partido_{id_}_plan_gobierno.pdf',
        'url': f'{BASE_URL}/Partido_{id_}_plan_gobierno.pdf',
    }
    for id_, partido, candidato in PARTIDOS
]

In [8]:
MANIFIESTO

[{'id_partido': 'A',
  'partido': 'Renovación Popular Social',
  'candidato': 'Candidata A',
  'archivo': 'Partido_A_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/main/planes_gobierno/Partido_A_plan_gobierno.pdf'},
 {'id_partido': 'B',
  'partido': 'Frente Progresista',
  'candidato': 'Candidato B',
  'archivo': 'Partido_B_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/main/planes_gobierno/Partido_B_plan_gobierno.pdf'},
 {'id_partido': 'C',
  'partido': 'Movimiento Democrático',
  'candidato': 'Candidata C',
  'archivo': 'Partido_C_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/main/planes_gobierno/Partido_C_plan_gobierno.pdf'},
 {'id_partido': 'D',
  'partido': 'Centro Cívico',
  'candidato': 'Candidato D',
  'archivo': 'Partido_D_plan_gobierno.pdf',
  'url': 'https://raw.githubusercontent.com/doctorado-cuanticp/text/main/planes_gobierno/Partido_D_plan_gobierno.p

## preTEXT-02: leer los PDFs y crear el corpus

Leemos cada PDF desde su enlace y lo abrimos en memoria con PyMuPDF. Unimos el texto de todas sus páginas y lo incorporamos al DataFrame `corpus`.

El resultado contiene **una fila por plan**, con su identificador, partido, candidatura y texto completo extraído en `text_raw`. Esa columna incluye todavía la portada y los títulos. No se guardan archivos en este paso.

In [9]:
import pymupdf as fitz
import pandas as pd

documents = []

for r in MANIFIESTO:
    with urlopen(r['url']) as response:
        contenido = response.read()

    with fitz.open(stream=contenido, filetype='pdf') as doc:
        texto = "\n".join(page.get_text() for page in doc)

    documents.append({
        'id_partido': r['id_partido'],
        'partido': r['partido'],
        'candidato': r['candidato'],
        'text_raw': texto,
    })

corpus = pd.DataFrame(documents)

corpus

,id_partido,partido,candidato,text_raw
0,A,Renovación Popular Social,Candidata A,PLAN DE GOBIERNO\nRenovación Popular Social\nC...
1,B,Frente Progresista,Candidato B,PLAN DE GOBIERNO\nFrente Progresista\nCandidat...
2,C,Movimiento Democrático,Candidata C,PLAN DE GOBIERNO\nMovimiento Democrático\nCand...
3,D,Centro Cívico,Candidato D,PLAN DE GOBIERNO\nCentro Cívico\nCandidatura: ...
4,E,Alianza Nacional,Candidata E,PLAN DE GOBIERNO\nAlianza Nacional\nCandidatur...
5,F,Libertad Republicana,Candidato F,PLAN DE GOBIERNO\nLibertad Republicana\nCandid...
6,G,Futuro Liberal,Candidata G,PLAN DE GOBIERNO\nFuturo Liberal\nCandidatura:...
7,H,Mercado y Libertad,Candidato H,PLAN DE GOBIERNO\nMercado y Libertad\nCandidat...


## preTEXT-03: limpiar artefactos, conservar lenguaje

Retiramos líneas exactas conocidas de estos PDFs: el título de portada, el aviso pedagógico, el nombre del partido, la candidatura y los cuatro títulos de sección. También normalizamos los espacios y eliminamos el carácter de guion opcional que puede aparecer en la extracción.

El resultado se guarda en una nueva columna, `texto`. Conservamos `text_raw` para comparar ambas versiones en el siguiente paso.

**Clave:** mantenemos las propuestas con sus mayúsculas, acentos, puntuación, negaciones y palabras funcionales. No eliminamos stopwords ni convertimos el texto en una bolsa de palabras. Esta limpieza corresponde a los documentos del ejercicio; si cambian los PDFs, hay que revisar qué líneas deben excluirse.

In [10]:
import re
import unicodedata

LINEAS_COMUNES = {
    'PLAN DE GOBIERNO',
    'Documento ficticio elaborado exclusivamente para fines pedagógicos.',
    'Diagnóstico y prioridades',
    'Economía y empleo',
    'Políticas sociales',
    'Instituciones y territorio',
}

def limpiar(fila):
    texto = unicodedata.normalize('NFC', fila['text_raw'])
    texto = texto.replace('\u00ad', '')

    excluir = LINEAS_COMUNES | {
        fila['partido'],
        f"Candidatura: {fila['candidato']}",
    }

    lineas = [
        linea.strip()
        for linea in texto.splitlines()
        if linea.strip() not in excluir
    ]

    return re.sub(r'\s+', ' ', ' '.join(lineas)).strip()


corpus['texto'] = corpus.apply(limpiar, axis=1)

corpus[['id_partido', 'texto']]

,id_partido,texto
0,A,Nuestro plan propone crecimiento económico con...
1,B,Nuestro plan propone crecimiento económico con...
2,C,Nuestro plan propone crecimiento económico sos...
3,D,Nuestro plan propone crecimiento económico sos...
4,E,"Nuestro plan propone crecimiento económico, em..."
5,F,"Nuestro plan propone crecimiento económico, em..."
6,G,"Nuestro plan propone crecimiento económico, em..."
7,H,"Nuestro plan propone crecimiento económico, em..."


## preTEXT-04: comparar el texto original y el preparado

Mostramos, para cada partido, el texto completo extraído (`text_raw`) y el texto preparado (`texto`). Como los planes son breves, podemos comparar ambas versiones sin recortarlas.

Compruebe que desaparecieron la portada y los títulos, pero se conservaron todas las propuestas. Preste atención a negaciones, puntuación y palabras que pudieron quedar separadas por saltos de línea. Si encuentra una diferencia dudosa, consulte el PDF original.

**Cuidado:** limpiar no significa mejorar automáticamente. La comparación permite comprobar qué información llegará al análisis.

In [11]:
for _, fila in corpus.iterrows():
    print('\n', fila['id_partido'], '—', fila['partido'])
    print('ORIGINAL:\n', fila['text_raw'])
    print('\nPREPARADO:\n', fila['texto'])
    print('-' * 80)


 A — Renovación Popular Social
ORIGINAL:
 PLAN DE GOBIERNO
Renovación Popular Social
Candidatura: Candidata A
Documento ficticio elaborado exclusivamente para fines pedagógicos.

Diagnóstico y prioridades
Nuestro plan propone crecimiento económico con empleo digno, seguridad ciudadana,
educación pública de calidad, salud universal y lucha contra la corrupción.
Impulsaremos vivienda social, descentralización, protección ambiental y servicios públicos
accesibles en todo el territorio.
Economía y empleo
El desarrollo nacional requiere un Estado activo que garantice derechos sociales, fortalezca
empresas públicas estratégicas y reduzca la desigualdad.
Políticas sociales
Promoveremos inversión pública, reforma tributaria progresiva, negociación colectiva,
protección laboral y mayor participación de trabajadores.
Instituciones y territorio
La inversión privada podrá contribuir al desarrollo cuando respete regulación, derechos
laborales y objetivos nacionales.


PREPARADO:
 Nuestro plan prop

## preTEXT-05: guardar el corpus preparado

Guardamos un único archivo: `planes_gobierno_preparados.csv`. Contiene una fila por plan y las columnas `id_partido`, `partido`, `candidato`, `text_raw` y `texto`.

Este CSV conecta los dos notebooks. En `TextAnalysis.ipynb` utilizaremos `texto` para construir los embeddings; `text_raw` quedará disponible para volver a comparar con la extracción original.

En Colab, el archivo se guarda en la carpeta de trabajo de la sesión. Descárguelo si desea conservarlo o cargarlo en otra sesión.

**Hasta aquí preparamos la evidencia. La evaluación de H1 y H2 viene después.**

In [12]:
corpus.to_csv(
    'planes_gobierno_preparados.csv',
    index=False,
    encoding='utf-8'
)

print('Corpus guardado. Continúe con TextAnalysis.ipynb.')

Corpus guardado. Continúe con TextAnalysis.ipynb.
